# Attenuator Calibration Lab Script

Dirty science-lab notebook for bridge-normalized attenuator calibration. It talks to the board only through `pcb.atten()` and `pcb.pd()` for acquisition; measure dark in the preceding cell before collecting data.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from math import erf
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.special import erfinv

import hispec_fibpcb as hspcb

BROKER = "hispec.caltech.edu"
LASER = "1028y"
PD_CHANNEL = "yj"
DWELL_S = 1
MAX_DAC_MV = hspcb.ATTENUATOR_DRIVE_MAX_MV
GAIN = hspcb.ATTENUATOR_DEFAULT_GAIN
ERF_SCALE = hspcb.ATTENUATOR_MODEL_ERF_SCALE

pcb = hspcb.HispecFibPcb(BROKER, connect=True)
pcb.status()

Measure dark first with `pd_dark`. Keep `persist=False` while exploring; switch to `True` only after the dark value has been reviewed.

In [ ]:
pcb.set_laser_level('1028y', 00)

In [ ]:
dark = pcb.pd_dark(PD_CHANNEL, duration_ms=2000, persist=False)
dark.dark

In [ ]:
pcb.pd_dark(PD_CHANNEL)

In [ ]:
pcb.set_laser_level('1028y', 50)

In [ ]:
@dataclass
class CalRecord:
    physical: str
    event: str
    sweep_mv: float
    other_mv: float
    net_mv: float
    raw_mv: float
    rms_mv: float
    scale: float
    tx: float
    usable: bool
    saturated: bool


def _pd_sample(channel: str = PD_CHANNEL) -> tuple[float, float, float]:
    pd = pcb.pd(channel)
    pd_channel = getattr(pd, channel)
    if pd_channel is None:
        raise RuntimeError(f"missing photodiode channel {channel}")
    window = pd_channel.window
    return (
        float(window.mean_net_mv),
        float(window.mean_mv),
        float(window.mean_net_err_mv),
    )


def _set_pair(laser: str, physical: str, sweep_mv: float, other_mv: float):
    sweep_mv = float(np.clip(sweep_mv, 0.0, MAX_DAC_MV))
    other_mv = float(np.clip(other_mv, 0.0, MAX_DAC_MV))
    if physical == "dac1":
        return pcb.atten(laser, value1_mv=sweep_mv, value2_mv=other_mv)
    if physical == "dac2":
        return pcb.atten(laser, value1_mv=other_mv, value2_mv=sweep_mv)
    raise ValueError("physical must be dac1 or dac2")


def measure_point(
    physical: str,
    sweep_mv: float,
    other_mv: float,
    *,
    event: str = "point",
    scale: float = 1.0,
    open_net_mv: float | None = None,
    limit: float = 5000
) -> CalRecord:
    
    _set_pair(LASER, physical, sweep_mv, other_mv)
    time.sleep(DWELL_S)
    net_mv, raw_mv, rms_mv = _pd_sample(PD_CHANNEL)
    
    saturated = raw_mv >= 5200 and net_mv > limit
    usable = (not saturated) and net_mv > rms_mv

    print(f'measuring {event} {physical}={sweep_mv}, other={other_mv} pd={net_mv}±{rms_mv} usable={usable} sat={saturated}')
    
    tx = np.nan
    if open_net_mv is not None and open_net_mv > 0.0 and scale > 0.0:
        tx = net_mv / (open_net_mv * scale)
        
    return CalRecord(physical, event, sweep_mv, other_mv, net_mv, raw_mv, rms_mv, scale, tx, usable, saturated)


def tx_to_delta(tx: np.ndarray | float) -> np.ndarray | float:
    erf_scale = erf(ERF_SCALE)
    return erfinv(erf_scale - 2.0 * erf_scale * np.asarray(tx))


def delta_to_tx(delta: np.ndarray | float) -> np.ndarray | float:
    erf_scale = erf(ERF_SCALE)
    return (erf_scale - np.vectorize(erf)(np.asarray(delta))) / (2.0 * erf_scale)

In [ ]:
def find_companion_start(physical: str, *, step_mv: float = 5.0) -> tuple[float, list[CalRecord]]:
    records: list[CalRecord] = []
    low, high = 0.0, MAX_DAC_MV
    for _ in range(16):
        mid = 0.5 * (low + high)
        rec = measure_point(physical, 0.0, mid, event="initial_probe", limit=4800)
        rec = measure_point(physical, 0.0, mid, event="initial_probe", limit=4800)
        records.append(rec)
        if rec.saturated:
            print(f'low {low}->{mid}')
            low = mid
        else:
            print(f'high {high}->{mid}')
            high = mid
        if high - low <= step_mv:
            break
    return high, records


def bridge_once(
    physical: str,
    sweep_mv: float,
    other_mv: float,
    open_net_mv: float,
    scale: float,
    records: list[CalRecord],
    *,
    step_mv: float = 5.0,
) -> tuple[float, float]:
    before = measure_point(physical, sweep_mv, other_mv, event="bridge_before", scale=scale, open_net_mv=open_net_mv)
    records.append(before)
    if not before.usable or before.net_mv <= 0.0:
        return other_mv, scale

    low, high = 0.0, other_mv
    for _ in range(16):
        mid = 0.5 * (low + high)
        probe = measure_point(physical, sweep_mv, mid, event="bridge_probe", scale=scale, open_net_mv=open_net_mv)
        records.append(probe)
        if probe.saturated:
            low = mid
        else:
            high = mid
        if high - low <= step_mv:
            break

    after = measure_point(physical, sweep_mv, high, event="bridge_after", scale=scale, open_net_mv=open_net_mv)
    if after.usable and after.net_mv > before.net_mv:
        scale *= after.net_mv / before.net_mv
        after.scale = scale
        after.tx = after.net_mv / (open_net_mv * scale)
        records.append(after)
        return high, scale

    records.append(after)
    return other_mv, scale


def acquire_physical(physical: str, *, step_mv: float = 50.0) -> list[CalRecord]:
    other_mv, records = find_companion_start(physical)
    reference = measure_point(physical, 0.0, other_mv, event="reference")
    records.append(reference)
    if not reference.usable:
        raise RuntimeError(f"{physical} open reference is not usable: {reference}")
    else:
        print(reference)
        
    open_net_mv = reference.net_mv
    scale = 1.0
    mv= 0.0
    while mv<=(MAX_DAC_MV+step_mv/2):
        rec = measure_point(physical, min(mv, MAX_DAC_MV), other_mv, scale=scale, open_net_mv=open_net_mv)
        records.append(rec)
        if rec.usable:
            mv += step_mv
            print(mv)
        else:
            print('rescale')
            other_mv, scale = bridge_once(physical, mv, other_mv, open_net_mv, scale, records)
            print(other_mv, scale)

    return records

# def acquire_physical(physical: str, *, step_mv: float = 5.0, max_records: int = 96) -> list[CalRecord]:
#     other_mv, records = find_companion_start(physical, step_mv=step_mv)
#     reference = measure_point(physical, 0.0, other_mv, event="reference")
#     records.append(reference)
#     if not reference.usable:
#         raise RuntimeError(f"{physical} open reference is not usable: {reference}")

#     open_net_mv = reference.net_mv
#     scale = 1.0
#     low, high = 0.0, MAX_DAC_MV
#     while len(records) < max_records and high - low > step_mv:
#         mid = 0.5 * (low + high)
#         rec = measure_point(physical, mid, other_mv, scale=scale, open_net_mv=open_net_mv)
#         records.append(rec)
#         if rec.usable:
#             low = mid
#             if MAX_DAC_MV - low <= step_mv:
#                 break
#         else:
#             high = mid
#         if high - low <= step_mv and high < MAX_DAC_MV - step_mv:
#             other_mv, scale = bridge_once(physical, low, other_mv, open_net_mv, scale, records, step_mv=step_mv)
#             high = MAX_DAC_MV
#     return records

In [ ]:
def fit_records(records: list[CalRecord], *, gain: float = GAIN) -> dict[str, float | int | bool]:
    fit_records = [r for r in records if r.usable and np.isfinite(r.tx) and 1e-10 < r.tx < 0.999999]
    if len(fit_records) < 6:
        raise RuntimeError(f"not enough fit records: {len(fit_records)}")
    x = gain * np.array([r.sweep_mv for r in fit_records], dtype=float)
    y = tx_to_delta(np.array([r.tx for r in fit_records], dtype=float))
    slope_inv, intercept = np.polyfit(x, y, 1)
    fvoa_50pct_mv = -intercept / slope_inv
    model_tx = delta_to_tx(slope_inv * x + intercept)
    measured_tx = np.array([r.tx for r in fit_records], dtype=float)
    residual_db = 10.0 * np.log10(np.clip(model_tx, 1e-300, np.inf) / np.clip(measured_tx, 1e-300, np.inf))
    return {
        "accepted": bool(np.isfinite(fvoa_50pct_mv) and fvoa_50pct_mv > 0.0 and slope_inv > 0.0),
        "points": len(fit_records),
        "fvoa_50pct_mv": float(fvoa_50pct_mv),
        "slope_inv_fvoa_mv": float(slope_inv),
        "gain": float(gain),
        "rms_db": float(np.sqrt(np.mean(residual_db * residual_db))),
        "max_abs_db": float(np.max(np.abs(residual_db))),
    }


def plot_records(records: list[CalRecord], fit: dict[str, float | int | bool]):
    sweep = np.array([r.sweep_mv for r in records], dtype=float)
    net = np.array([r.net_mv for r in records], dtype=float)
    tx = np.array([r.tx for r in records], dtype=float)
    usable = np.array([r.usable for r in records], dtype=bool)
    fig, axes = plt.subplots(2, 1, figsize=(8.0, 7.0), sharex=True)
    axes[0].scatter(sweep, net, c=np.where(usable, "tab:blue", "tab:red"), s=18)
    axes[0].set_ylabel(f"{PD_CHANNEL}_net (mV)")
    axes[1].scatter(sweep[usable], tx[usable], color="tab:blue", s=18)
    grid = np.linspace(0.0, MAX_DAC_MV, 400)
    delta = fit["slope_inv_fvoa_mv"] * ((fit["gain"] * grid) - fit["fvoa_50pct_mv"])
    axes[1].plot(grid, delta_to_tx(delta), color="tab:orange")
    axes[1].set_xlabel("swept DAC (mV)")
    axes[1].set_ylabel("relative transmission")
    fig.tight_layout()
    return fig

In [ ]:
dac1_records = acquire_physical("dac1", step_mv=100)
dac1_fit = fit_records(dac1_records)
dac1_fit

In [ ]:
plot_records(dac1_records, dac1_fit)

In [ ]:
dac2_records = acquire_physical("dac2")
dac2_fit = fit_records(dac2_records)
dac2_fit

In [ ]:
plot_records(dac2_records, dac2_fit)

In [ ]:
records = [asdict(r) for r in dac1_records + dac2_records]
coeff = {
    "dac1": {k: dac1_fit[k] for k in ("fvoa_50pct_mv", "slope_inv_fvoa_mv", "gain")},
    "dac2": {k: dac2_fit[k] for k in ("fvoa_50pct_mv", "slope_inv_fvoa_mv", "gain")},
}
coeff

Review the plots and coefficients before applying. Keep `persist=False` until the calibration has been repeated and accepted.

In [ ]:
# pcb.set_atten_coeff(LASER, coeff["dac1"], coeff["dac2"], persist=False)